# Adding Calculated Fields

## Overview
In this lesson we will download a small example TEEHR Evaluation from S3 and work through the different ways to add calculated fields to the joined_timeseries table, save them to the joined_timeseries table on disk to persist them, or add them temporarily before calculating metrics.  The first few steps for creating and cloning the TEEHR Evaluation should look familiar if you worked through the previous examples.

### Create a new Evaluation
First we will import TEEHR along with some other required libraries for this example.  Then we create a new instance of the Evaluation that points to a directory where the evaluation data will be stored.

In [1]:
import teehr
from pathlib import Path
import shutil
from teehr.example_data.setup_e0_2_example import download_e0_2_example

# Tell Bokeh to output plots in the notebook
from bokeh.io import output_notebook
output_notebook()

Loading BokehJS ...

In [2]:
# Define the directory where the Evaluation will be created
test_eval_dir = Path(Path().home(), "temp", "08_calculated_fields")
shutil.rmtree(test_eval_dir, ignore_errors=True)

download_e0_2_example(temp_dir=test_eval_dir)

# Create an Evaluation object and create the directory
ev = teehr.LocalReadWriteEvaluation(dir_path=Path(test_eval_dir, "e0_2_location_example"), create_dir=True)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region


✅ Downloaded to /Users/mdenno/temp/08_calculated_fields/e0_2_location_example.tar.gz
Extracting archive...
✅ Extraction complete
✅ Removed archive /Users/mdenno/temp/08_calculated_fields/e0_2_location_example.tar.gz


INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS credentials from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
:: loading settings :: url = jar:file:/Users/mdenno/repos/teehr/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mdenno/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mdenno/.ivy2.5.2/jars
org.apache.sedona#sedona-spark-shaded-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.iceberg#iceberg-spark-extensions-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a depen

## Add Calculated Fields To Use In Metric Calculations

Now we will get right to it.  Lets start by taking a look at the `joined_timeseries_view` that joins the primary and secondary timeseries. Lets see what fields are included.

In [3]:
ev.joined_timeseries_view().to_sdf().show(n=10, truncate=False)

INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: variables.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: variables.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.variables.
IN

+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+
|reference_time|value_time         |primary_location_id|secondary_location_id|primary_value|secondary_value|configuration_name |unit_name|variable_name         |member|
+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly

Ok, so, as shown, the `joined_timeseries_view` now only contains the basic fields required without any attributes or user defined fields. Now lets add some fields to the `joined_timeseries_view` using the TEEHR calculated fields.  Adding fields can be useful for grouping and filtering operations, which allow you to define the population when calculating metrics.  The TEEHR calculated fields (CF) allow for row level and timeseries aware CFs:

* Row level CFs are applied to each row in the table based on data that is in one or more existing fields.  These are applied per row and are not aware of the data in any other row (e.g., are not aware of any other timeseries values in a "timeseries").  This can be used for adding fields such as a field based on the data/time (e.g., month, year, season, etc.) or based on the value field (e.g., normalized flow, log flow, etc.) and many other uses.
* Timeseries aware CFs are aware of ordered groups of data (e.g., a timeseries).  This is useful for things such as event detection, base flow separation, and other fields that need to be calculated based on a entire collection of timeseries values.  The definition of what creates a unique set of timeseries (i.e., a timeseries) can be specified.

There are two ways that these CFs can be used:  

* First, they can be used to add the CF to the `joined_timeseries_view` which can then be persisted by writing to disk. This is useful if the calculation is expected to be needed for multiple different metric calculations.
* Second, they can be used as a pre-processing step in the calculation of metrics.

These use cases will be demonstrated below.  First we will import the CF classes.  Normally this would be done at the top of the page, but is done here for demonstration purposes.

In [4]:
# first we need to import the UDF classes
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf

### Available Calculated Fields
There are a number of calculated field classes that are included in the TEEHR package.  They are:

Row Level Calculated Fields:
- Month
- Year
- WaterYear
- NormalizedFlow
- Seasons
- ForecastLeadTime
- ForecastLeadTimeBins
- ThresholdValueExceeded
- DayOfYear

Timeseries Aware Calculated Fields.
- AbovePercentileEventDetection

There will be more added over time.  If there is one you are particularly interested in, please reach out and let us know.

### Add Calculated Field in Memory
Now we will use the Row level CFs to add year, month, water year and season to the `joined_timeseries` table in memory, using the `add_udf_columns()` method on the `joined_timeseries` table, but will not save it to disk yet.  Adding the UDFs and displaying the table shows that the new fields were added.

In [5]:
sdf = ev.joined_timeseries_view().add_calculated_fields([
    rcf.Month(),
    rcf.Year(),
    rcf.WaterYear(),
    rcf.Seasons()
]).to_sdf()
sdf.show(n=10, truncate=False)

INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: variables.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: variables.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.variables.
IN

+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+-----+----+----------+------+
|reference_time|value_time         |primary_location_id|secondary_location_id|primary_value|secondary_value|configuration_name |unit_name|variable_name         |member|month|year|water_year|season|
+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+-----+----+----------+------+
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |10   |2000|2001      |fall  |
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |10   |2000|2001      |fall  |
|NULL     

However, if we query the view again, we can see that the additional fields are not there.  This is because we did not write the view with the additional fields to disk.

In [6]:
ev.joined_timeseries_view().to_sdf().show(n=10, truncate=False)

INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: variables.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: variables.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.variables.
IN

+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+
|reference_time|value_time         |primary_location_id|secondary_location_id|primary_value|secondary_value|configuration_name |unit_name|variable_name         |member|
+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly

### Add Calculated Field and Persist
Now we will add the UDFs again, but this time we will write the table with the additional fields to disk so they are persisted.

In [7]:
ev.joined_timeseries_view().add_calculated_fields([
    rcf.Month(),
    rcf.Year(),
    rcf.WaterYear(),
    rcf.Seasons()
]).write(table_name="joined_timeseries")


INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.primary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: variables.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: variables.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.variables.
IN

And query the table again.  This time we can see that the new fields we added are there as they were written to disk and are now part of the `joined_timeseries` table on disk.

In [8]:
ev.table("joined_timeseries").to_sdf().show(n=10, truncate=False)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.


+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+-----+----+----------+------+--------------------------+--------------------------+
|reference_time|value_time         |primary_location_id|secondary_location_id|primary_value|secondary_value|configuration_name |unit_name|variable_name         |member|month|year|water_year|season|created_at                |updated_at                |
+--------------+-------------------+-------------------+---------------------+-------------+---------------+-------------------+---------+----------------------+------+-----+----+----------+------+--------------------------+--------------------------+
|NULL          |2000-10-01 09:00:00|usgs-14138800      |nwm30-23736071       |9.825946     |0.07           |nwm30_retrospective|m^3/s    |streamflow_hourly_inst|NULL  |10   |2000|2001      |fall  |2026-05-16 12:31:01.308197|2026-05-16 12:31:01.

### Timeseries Aware Calculated Fields
The timeseries aware calculated fields behave the same way from the users perspective, but behind the scenes are performing some extra grouping and sorting to ensure that the field is calculated based on an ordered group of timeseries values (i.e., a "timeseries").  This is necessary for doing things like event detection, but comes at a computational cost, so use with care, especially on large datasets.  Lets try it.  This time we will jump right to writing the resulting data frame back to disk to persist it, but you could add the field and display the results without persisting as we did above.

In [9]:
ev.table("joined_timeseries").add_calculated_fields([
    tcf.AbovePercentileEventDetection()
]).write(table_name="joined_timeseries")

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
/var/folders/lq/jxwkhql53zd876vpfycn48480000gn/T/ipykernel_94879/1490925760.py:1: DeprecationWarning: write() is deprecated, use write_to() instead.
  ev.table("joined_timeseries").add_calculated_fields([
INFO:teehr.evaluation.dataframe_base:Writing to table: joined_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.write:Start writing to warehouse 

And query the `joined_timeseries` table to see the new `event` and `event_id` fields.

In [10]:
ev.table("joined_timeseries").to_sdf().show(n=10, truncate=False)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.


+--------------+-------------------+-------------------+----------------------+---------+-------------------+---------------------+-------------+---------------+------+-----+----+----------+------+-----------+---------------------------------------+--------------------------+--------------------------+
|reference_time|primary_location_id|configuration_name |variable_name         |unit_name|value_time         |secondary_location_id|primary_value|secondary_value|member|month|year|water_year|season|event_above|event_above_id                         |created_at                |updated_at                |
+--------------+-------------------+-------------------+----------------------+---------+-------------------+---------------------+-------------+---------------+------+-----+----+----------+------+-----------+---------------------------------------+--------------------------+--------------------------+
|NULL          |usgs-14316700      |nwm30_retrospective|streamflow_hourly_inst|m^3/s    

In the table above you can see the `event` and `event_id` fields, but it is a bit difficult to see what was done based on the table alone.  Lets create a plot to see what the new fields mean.  In the next few cells we will query the `joined_timeseries` table and filter the data to a single location ('usgs-14138800') and only values that were identified as `event=True`, then create a plot where we color the points by the new `event_id` field.

In [11]:
import hvplot.pandas  # noqa

In [12]:
pdf = ev.table("joined_timeseries").filter([
    "primary_location_id = 'usgs-14138800'",
    "event_above = true",
]).to_pandas()

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter ["primary_location_id = 'usgs-14138800'", 'event_above = true'].


In [13]:
primary_plot = pdf.hvplot.points(x="value_time", y="primary_value", color="event_above_id") #.opts(width=1200, height=400)

In [14]:
primary_plot.opts(width=1200, height=400)

:Points   [value_time,primary_value]   (event_above_id)

If you zoom in on the plot above, you can see that each unique `event` has been given a unique `event_id` that was used to color the individual data points.  This was done by first identifying the 85 percentile value at each location and identifying each value as either above or below that value for its location.  Then the values were grouped by continuous sets of values that were identified as `event=True` and each continuous group was given an `event_id`.  This is just one example of what can be done using `TimeseriesAwareCalculatedFields`.

## Use Calculated Fields in Metrics

Beyond just identifying high flow events (in this example), the `event` and `event_id` fields can be used in subsequent metrics calculations.  In the next few cells we will demonstrate one way that the `event` and `event_id` fields can be used by working through the steps to calculate the `event_max_relative_bias`.  The `event_max_relative_bias` is the relative bias between the primary and secondary timeseries maximum values within each event.  We calculate this in twos steps using chained queries.  We will do it in two steps to demonstrate what we are doing. Note, if you have not worked though the grouping and filter notebooks you may want to go back and do that first as it is an important concept to understanding what is being done here. First we run a metrics query where we filter to a single location and only values that were identified as being `event=True`, group by `configuration_name`, `primary_location_id` and `event_id`, and calculate the maximum primary and secondary values which we call `max_primary_value` and `max_secondary_value` but could give them any name we wanted. 

In [15]:
(
    ev.table("joined_timeseries")
    .filter([
        "primary_location_id = 'usgs-14138800'",
        "event_above = true"
    ])
    .aggregate(
        group_by=["configuration_name", "primary_location_id", "event_above_id"],
        metrics=[
            teehr.Signatures.Maximum(
                # input_field_names=["primary_value"],
                primary_field_name="primary_value",
                output_field_name="max_primary_value"
            ),
            teehr.Signatures.Maximum(
                # input_field_names=["secondary_value"],
                primary_field_name="secondary_value",
                output_field_name="max_secondary_value"
            )
        ]
    )
    .to_sdf().show(n=10, truncate=False)
)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter ["primary_location_id = 'usgs-14138800'", 'event_above = true'].
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


+-------------------+-------------------+---------------------------------------+------------------+-------------------+
|configuration_name |primary_location_id|event_above_id                         |max_primary_value |max_secondary_value|
+-------------------+-------------------+---------------------------------------+------------------+-------------------+
|nwm30_retrospective|usgs-14138800      |2002-12-15 08:00:00-2002-12-15 14:00:00|3.0015857219696045|3.6599998474121094 |
|nwm30_retrospective|usgs-14138800      |2003-12-05 12:00:00-2003-12-07 16:00:00|6.399607181549072 |8.84999942779541   |
|nwm30_retrospective|usgs-14138800      |2006-01-07 13:00:00-2006-01-08 18:00:00|4.3041605949401855|14.389999389648438 |
|nwm30_retrospective|usgs-14138800      |2003-12-12 16:00:00-2003-12-15 14:00:00|20.189910888671875|10.729999542236328 |
|nwm30_retrospective|usgs-14138800      |2011-05-11 01:00:00-2011-05-11 10:00:00|2.8883183002471924|4.71999979019165   |
|nwm30_retrospective|usgs-141388

You can see that this gives us the `max_primary_value` and `max_secondary_value`  for each unique group of `configuration_name`, `primary_location_id` and `event_id`.  But, that is not what we are actually trying to calculate.  We are really trying to calculate the `event_max_relative_bias`.  To do that we have to add one more step by chaining together queries.  In the following query we add an additional chained query where we only group by `configuration_name` and `primary_location_id` which causes the query to aggregate the values from the different events, and then as an aggregation method (`metrics`) we choose relative bias.

In [16]:
(
    ev.table("joined_timeseries")
    .filter([
        "primary_location_id = 'usgs-14138800'",
        "event_above = true"
    ])
    .aggregate(
        group_by=["configuration_name", "primary_location_id", "event_above_id"],
        metrics=[
            teehr.Signatures.Maximum(
                primary_field_name="primary_value",
                output_field_name="max_primary_value"
            ),
            teehr.Signatures.Maximum(
                primary_field_name="secondary_value",
                output_field_name="max_secondary_value"
            )
        ]
    )
    .aggregate(
        group_by=["configuration_name", "primary_location_id"],
        metrics=[
            teehr.DeterministicMetrics.RelativeBias(
                primary_field_name="max_primary_value",
                secondary_field_name="max_secondary_value",
                output_field_name="event_max_relative_bias"
            )
        ]
    )
    .to_sdf().show(n=10, truncate=False)
)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter ["primary_location_id = 'usgs-14138800'", 'event_above = true'].
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


+-------------------+-------------------+-----------------------+
|configuration_name |primary_location_id|event_max_relative_bias|
+-------------------+-------------------+-----------------------+
|nwm30_retrospective|usgs-14138800      |-0.08458244928652466   |
+-------------------+-------------------+-----------------------+



One last thing to cover here.  So far we have added the calculated fields on the `joined_timeseries` table, written them to disk, and then queried the `joined_timeseries` table to calculate metrics.  This works well and allows the calculated fields to be calculated once and used in many subsequent metrics, plots, etc.  However, you may wish to add temporary fields to the `joined_timeseries` table as part of the metrics calculation.  This can be done too.  Building on the previous example, where we calculated the "event_max_relative_bias", lets now assume we want to calculate the same metric but for the 90th percentile instead of the default 85th percentile that we used when we added the added the `event` and `event_id` fields to the `joined_timeseries` table.  We could add the new "90th percentile event" to the `joined_timeseries` table and save to disk and then proceed as we did before, or we can add new `event90` and `event90_id` fields to the data frame temporarily before calculating the maximum event values and ultimately the "event_90th_max_relative_bias".

In [17]:
(
    ev.table("joined_timeseries")
    # Add the AbovePercentileEventDetection calculated field to identify events greater than the 90th percentile.
    # Note the output_event_field_name and output_event_id_field_name are set to "event90" and "event90_id" respectively.
    .add_calculated_fields([
        tcf.AbovePercentileEventDetection(
            quantile=0.90,
            output_event_field_name="event90",
            output_event_id_field_name="event90_id"
        )
    ])
    .filter([
        "primary_location_id = 'usgs-14138800'",
        "event90 = true",
    ])
    # First query to calculate the maximum primary and secondary values for each event.
    # Note the filters are set to only include events where event90 is true and the group_by includes event90_id.
    .aggregate(
        group_by=["configuration_name", "primary_location_id", "event90_id"],
        metrics=[
            teehr.Signatures.Maximum(
                primary_field_name="primary_value",
                output_field_name="max_primary_value"
            ),
            teehr.Signatures.Maximum(
                primary_field_name="secondary_value",
                output_field_name="max_secondary_value"
            )
        ]
    )
    # Second query to calculate the relative bias between the maximum primary and secondary values.
    .aggregate(
        group_by=["configuration_name", "primary_location_id"],
        metrics=[
            teehr.DeterministicMetrics.RelativeBias(
                primary_field_name="max_primary_value",
                secondary_field_name="max_secondary_value",
                output_field_name="event_90th_max_relative_bias"
            )
        ]
    )
    # Convert the metrics to a pandas DataFrame
    .to_sdf().show(n=10, truncate=False)
)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter ["primary_location_id = 'usgs-14138800'", 'event90 = true'].
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


+-------------------+-------------------+----------------------------+
|configuration_name |primary_location_id|event_90th_max_relative_bias|
+-------------------+-------------------+----------------------------+
|nwm30_retrospective|usgs-14138800      |-0.087357269094153          |
+-------------------+-------------------+----------------------------+



Categorical Deterministic Metrics
---------------------------------

Now that we understand routines for grouping/filtering data and added calculated fields in TEEHR to obtain performance metrics, we can introduce the concept of categorical deterministic metrics. 

Categorical deterministic metrics are utilized to measure qualitative attributes and are used to evaluate binary outcomes given some condition or categorical classification. Put simply, these methods help compare predictions made by the model with observed data to indicate where the model was right or wrong.

Currently, the following categorical deterministic methods are available in TEEHR:

- [ConfusionMatrix](https://rtiinternational.github.io/teehr/api/generated/teehr.DeterministicMetrics.html#teehr.DeterministicMetrics.ConfusionMatrix)
- [FalseAlarmRatio](https://rtiinternational.github.io/teehr/api/generated/teehr.DeterministicMetrics.html#teehr.DeterministicMetrics.FalseAlarmRatio)
- [ProbabilityOfDetection](https://rtiinternational.github.io/teehr/api/generated/teehr.DeterministicMetrics.html#teehr.DeterministicMetrics.ProbabilityOfDetection)
- [ProbabilityOfFalseDetection](https://rtiinternational.github.io/teehr/api/generated/teehr.DeterministicMetrics.html#teehr.DeterministicMetrics.ProbabilityOfFalseDetection)
- [CriticalSuccessIndex](https://rtiinternational.github.io/teehr/api/generated/teehr.DeterministicMetrics.html#teehr.DeterministicMetrics.CriticalSuccessIndex)

Unlike other metrics in TEEHR which operate on the default fields present in the joined timeseries table, categorical deterministic metrics require an additional 'threshold field' column to categorize model predictive performance based on a user defined flow threshold. 

For example, let's utilize our categorical metrics to evaluate model performance when predicting streamflow that exceeds the 10% return interval. Lets start from the default `joined_timeseries` table and complete the chained operation in-memory as opposed to writing to disk or using the fields we calculated in the previous steps.

In [18]:
# overwrite the existing joined_timeseries table to demonstrate chained query
ev.joined_timeseries_view().write(table_name="joined_timeseries", write_mode="create_or_replace")

/var/folders/lq/jxwkhql53zd876vpfycn48480000gn/T/ipykernel_94879/3920594018.py:2: DeprecationWarning: write() is deprecated, use write_to() instead.
  ev.joined_timeseries_view().write(table_name="joined_timeseries", write_mode="create_or_replace")
INFO:teehr.evaluation.dataframe_base:Writing to table: joined_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.views.joined_timeseries_view:Computing joined timeseries view
INFO:teehr.evaluation.tables.generic_table:Getting table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: primary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.primary_timeseries.
I

In [19]:
metrics_df = ev.table("joined_timeseries").add_calculated_fields([
    # adds 'event', 'event_id', and 'quantile_value' fields to the joined timeseries table
    teehr.TimeseriesAwareCalculatedFields.AbovePercentileEventDetection(
        quantile=0.90,
        add_quantile_field=True
    )
]).aggregate(
    # calculate all available categorical deterministic metrics using the 'quantile_value' field as the threshold
    group_by=['primary_location_id', 'configuration_name'],
    metrics=[
        teehr.DeterministicMetrics.ConfusionMatrix(
            threshold_field_name='quantile_value'
        ),
        teehr.DeterministicMetrics.FalseAlarmRatio(
            threshold_field_name='quantile_value'
        ),
        teehr.DeterministicMetrics.ProbabilityOfDetection(
            threshold_field_name='quantile_value'
        ),
        teehr.DeterministicMetrics.ProbabilityOfFalseDetection(
            threshold_field_name='quantile_value'
        ),
        teehr.DeterministicMetrics.CriticalSuccessIndex(
            threshold_field_name='quantile_value'
        )
    ]
).to_sdf().show(n=10, truncate=False)

INFO:teehr.evaluation.tables.generic_table:Getting table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.read:Reading files from local.teehr.joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.


+-------------------+-------------------+-----------------------------------------------------+------------------+------------------------+------------------------------+----------------------+
|primary_location_id|configuration_name |confusion_matrix                                     |false_alarm_ratio |probability_of_detection|probability_of_false_detection|critical_success_index|
+-------------------+-------------------+-----------------------------------------------------+------------------+------------------------+------------------------------+----------------------+
|usgs-14138800      |nwm30_retrospective|{TP -> 24504, TN -> 348200, FP -> 12340, FN -> 16068}|0.3349256323960482|0.6039633244602188      |0.034226438120596885          |0.46310855760508013   |
|usgs-14316700      |nwm30_retrospective|{TP -> 25204, TN -> 345988, FP -> 13892, FN -> 15204}|0.3553304685901371|0.6237378736883785      |0.03860175614093587           |0.4641620626151013    |
+-------------------+---------

In the above example, we utilize a chained query to add a 'threshold field' to our unaltered `joined_timeseries` table and calculate all available categorical metrics in one operation. 

The chained operation executes in two steps:

- The first portion of the operation uses the `AbovePercentileEventDetection` method with `add_quantile_field` set to True to obtain the flow value that corresponds to the 90% quantile for each row in the joined timeseries table.

- The following portion takes the resulting table (with `AbovePercentileEventDetection` fields present) and executes a metrics query for our categorical deterministic metrics (with `threshold_field_name` arguments populated with the default `output_quantile_field_name` specified in the `AbovePercentileEventDetection` method). 

As you can see, we obtain a unique entry for the following metrics per unique combination of `primary_location_id` and `configuration_name` (as specified in the `group_by` argument in the `ev.table.aggregate()` call).

<b>When employing categorical deterministic metrics in TEEHR, keep in mind that each aggregation must correspond to exactly one 'threshold value'</b>. 

In this example, that criteria is achieved inherently given the `quantile_value` was generated using a `TimeSeriesAwareCalculatedField` which considers each 'unique timeseries' (where 'unique timeseries' is defined by `AbovePercentileEventDetection.uniqueness_fields`, by default=`['reference_time', 'primary_location_id', 'configuration_name', 'variable_name', 'unit_name']`). 

By that same logic, each unique combination of `primary_location_id` and `configuration_name` defined in the `group_by` argument in this example has exactly one 'threshold value' because `['reference_time', 'variable_name', 'unit_name']` fields are constant for each unique combination of `['primary_location_id, 'configuration_name']`.

In [20]:
ev.spark.stop()